# Transfer Learning — ECG Classifier

Load the best checkpoint produced by `ecg_kaggle_early_stopping_checkpointed.ipynb` and extract the underlying PyTorch model for inspection or transfer learning.

## 1. Import Required Libraries

In [1]:
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import lr_scheduler

import pytorch_lightning as pl
import torchmetrics

## 2. Define Model Architecture

These class definitions must match the ones used during training exactly — they are required to deserialise the checkpoint.

In [2]:
class EcgClassifier(nn.Module):
    def __init__(self):
        super(EcgClassifier, self).__init__()

        self.conv1 = nn.Conv1d(in_channels=1, out_channels=32, kernel_size=5)
        self.conv2 = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=5)
        self.pool = nn.MaxPool1d(kernel_size=3)
        self.conv3 = nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3)
        self.conv4 = nn.Conv1d(in_channels=128, out_channels=256, kernel_size=3)
        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(0.5)

        self.fc1 = nn.Linear(256, 1024)
        self.fc2 = nn.Linear(1024, 256)
        self.fc3 = nn.Linear(256, 32)
        self.fc4 = nn.Linear(32, 5)

        self.relu = nn.ReLU()
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        # Input shape: (batch, 187) -> add channel dim -> (batch, 1, 187)
        x = x.unsqueeze(1)

        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.pool(x)
        x = self.relu(self.conv3(x))
        x = self.relu(self.conv4(x))
        x = self.global_avg_pool(x)
        x = self.dropout(x)

        x = x.view(x.size(0), -1)  # Flatten

        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.relu(self.fc3(x))
        x = self.dropout(x)
        x = self.fc4(x)
        # softmax omitted — CrossEntropyLoss applies it internally

        return x

In [3]:
class EcgLightningModule(pl.LightningModule):
    def __init__(self, num_classes: int, learning_rate: float = 1e-3, weight_decay: float = 1e-4,
                 class_weights: torch.Tensor = None):
        super(EcgLightningModule, self).__init__()
        self.save_hyperparameters(ignore=["class_weights"])
        self.model = EcgClassifier()
        self.num_classes = num_classes
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.register_buffer("class_weights", class_weights)

        self.training_step_outputs = {"loss": [], "acc": []}
        self.validation_step_outputs = {"loss": [], "acc": []}

        self.train_loss = torchmetrics.MeanMetric()
        self.val_loss   = torchmetrics.MeanMetric()
        self.train_acc  = torchmetrics.classification.Accuracy(num_classes=num_classes, task="multiclass")
        self.val_acc    = torchmetrics.classification.Accuracy(num_classes=num_classes, task="multiclass")

    @property
    def loss_fn(self):
        return nn.CrossEntropyLoss(weight=self.class_weights)

    def forward(self, x):
        return self.model(x)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            params=self.model.parameters(),
            lr=self.learning_rate,
            weight_decay=self.weight_decay,
        )
        scheduler = lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=2, min_lr=1e-6,
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {"scheduler": scheduler, "monitor": "val_loss", "interval": "epoch", "frequency": 1},
        }

    def predict_step(self, batch, batch_idx):
        x, _ = batch
        return self.model(x).argmax(dim=1)

## 3. Load Best Checkpoint from Lightning Logs

Find the most recently modified `.ckpt` file under `lightning_logs/` and load it.

In [4]:
notebook_dir = Path(".")

# Find all checkpoints under lightning_logs and pick the most recently modified one
ckpt_files = sorted(
    notebook_dir.glob("lightning_logs/**/checkpoints/*.ckpt"),
    key=lambda p: p.stat().st_mtime,
)

if not ckpt_files:
    raise FileNotFoundError("No checkpoints found under lightning_logs/. Run training first.")

ckpt_path = ckpt_files[-1]
print(f"Loading checkpoint: {ckpt_path}")

# Load the full Lightning module from the checkpoint.
# strict=False is required because class_weights is a registered buffer saved in the
# checkpoint state_dict, but is None (and thus absent) in the freshly constructed module.
# It is only needed for training loss — not for inference.
lightning_module = EcgLightningModule.load_from_checkpoint(
    ckpt_path,
    num_classes=5,
    strict=False,
)

# Extract just the inner nn.Module and set to eval mode
model: EcgClassifier = lightning_module.model
model.eval()

print("Model loaded and set to eval() mode.")

Loading checkpoint: lightning_logs/version_49/checkpoints/ecg-best-epoch=23-val_loss=0.3942.ckpt
Model loaded and set to eval() mode.


/home/michaelw/.pyenv/versions/py_3_12_8/lib/python3.12/site-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['class_weights']


## 4. Inspect Model Structure

In [5]:
# Print the PyTorch module tree
print(model)

EcgClassifier(
  (conv1): Conv1d(1, 32, kernel_size=(5,), stride=(1,))
  (conv2): Conv1d(32, 64, kernel_size=(5,), stride=(1,))
  (pool): MaxPool1d(kernel_size=3, stride=3, padding=0, dilation=1, ceil_mode=False)
  (conv3): Conv1d(64, 128, kernel_size=(3,), stride=(1,))
  (conv4): Conv1d(128, 256, kernel_size=(3,), stride=(1,))
  (global_avg_pool): AdaptiveAvgPool1d(output_size=1)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc1): Linear(in_features=256, out_features=1024, bias=True)
  (fc2): Linear(in_features=1024, out_features=256, bias=True)
  (fc3): Linear(in_features=256, out_features=32, bias=True)
  (fc4): Linear(in_features=32, out_features=5, bias=True)
  (relu): ReLU()
  (softmax): Softmax(dim=1)
)


In [ ]:
import traceback

try:
    from torchinfo import summary
    model.cpu()
    # Pass an explicit dummy input so torchinfo doesn't have to guess the shape.
    # The model expects (batch, 187) flat signals; unsqueeze(1) happens internally.
    dummy = torch.zeros(1, 187)
    result = summary(
        model,
        input_data=dummy,
        col_names=["input_size", "output_size", "num_params", "trainable"],
        verbose=0,
    )
    print(result)https://github.com/MichaelJWebster/arrhythmia_detection_ecg
except ImportError:
    print("torchinfo not installed — run `pip install torchinfo` for a detailed layer summary.")
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters    : {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
except Exception:
    traceback.print_exc()
    total_params = sum(p.numel() for p in model.parameters())
    print(f"\nTotal parameters    : {total_params:,}")

Layer (type:depth-idx)                   Input Shape               Output Shape              Param #                   Trainable
EcgClassifier                            [1, 187]                  [1, 5]                    --                        True
├─Conv1d: 1-1                            [1, 1, 187]               [1, 32, 183]              192                       True
├─ReLU: 1-2                              [1, 32, 183]              [1, 32, 183]              --                        --
├─Conv1d: 1-3                            [1, 32, 183]              [1, 64, 179]              10,304                    True
├─ReLU: 1-4                              [1, 64, 179]              [1, 64, 179]              --                        --
├─MaxPool1d: 1-5                         [1, 64, 179]              [1, 64, 59]               --                        --
├─Conv1d: 1-6                            [1, 64, 59]               [1, 128, 57]              24,704                    True
├─ReLU: 1